# Zutaten-Normalisierung & Gruppierung mit NLP

## Zielsetzung
Erstellung eines Ähnlichkeits-Caches für Zutaten aus TheMealDB, um:
- **Plural/Singular** zusammenzufassen (z.B. "Tomato" ↔ "Tomatoes")
- **Varianten** zu erkennen (z.B. "Cherry Tomatoes" ↔ "Baby Plum Tomatoes")
- **Modifikatoren** zu behandeln (z.B. "Fresh Basil" → "Basil")

---

## Workflow-Übersicht

```
1. Zutaten-Pool laden
    ↓
2. Grammatik-Analyse (spaCy)
    ↓
3. Token-Cache erstellen
    ↓
4. Ähnlichkeits-Vergleiche
    ↓
5. JSON-Export
```

---

## 1. Grammatik-Analyse mit spaCy

### Kernproblem: "Tomato Sauce" vs. "Tomatoes"
- **HEAD**: Grammatikalisches Zentrum (→ `sauce` / `tomato`)
- **MODIFIER**: Erweiterte Information (→ `tomato` / ∅)

### Manuelle Korrekturen
```python
NOISE_WORDS = {"leaves", "leaf", "seed", "nuts", "yolks"}
MANUAL_CORRECTIONS = {"leaves": "leaf", "nuts": "nut", "yolks": "yolk"}
```

### Funktion: `analyze_grammar(text)`
- Lemmatisierung (z.B. "tomatoes" → "tomato")
- Trennung von HEAD und MODIFIERS
- Ignoriert: Adjektive (`amod`), Präpositionen (`prep`)

**Beispiel:**
```
"Fresh Cherry Tomatoes"
→ HEAD: "tomato"
→ MODIFIER: "cherry"
(Ignoriert: "fresh" als Adjektiv)
```

---

## 2. Token-Vektorisierung

### Vektorkombination (4 Methoden)
Jede Zutat wird in 2 Vektoren zerlegt (HEAD + MODIFIER) und kombiniert:

| Methode | Formel | Vorteil |
|---------|--------|---------|
| `weighted` | 70% HEAD + 30% MOD | Betont grammatikalische Hierarchie |
| `concat` | [HEAD \|\| MOD] | Keine Informationsverlust (600D) |
| `hadamard` | HEAD ⊙ MOD | Betont gemeinsame Features |
| `max` | max(HEAD, MOD) | Robusteste Features beider Vektoren |

**Aktuell genutzt:** `METHOD = "max"`

---

## 3. IngredientTokenCache

### Initialisierung
```python
cache = IngredientTokenCache(ALL_INGREDIENTS, method="max")
```

**Workflow:**
1. Für jede Zutat: `analyze_grammar()` → HEAD + MOD
2. Konvertierung zu spaCy-Tokens (`en_core_web_md` für Vektoren)
3. Kombination zu `combine_tokens[ingredient]`

**Vorteil:** Einmalige Berechnung statt 877² Vergleiche!

---

## 4. Ähnlichkeits-Vergleich

### Funktion: `check_similarity_combined()`
- **Cosinus-Ähnlichkeit** zwischen kombinierten Vektoren
- Threshold: `0.85` (anpassbar)
- Output: `[(ingredient, score), ...]` (sortiert nach Score)

**Beispiel:**
```python
cache.check_similarity("Cucumber", threshold=0.85)
# → {"Cucumber": [("Cucumbers", 0.98), ("Zucchini", 0.87), ...]}
```

---

## 5. JSON-Export

### Dateiformat
```json
{
  "Chicken": [
     ["Chicken Breast", 0.95],
     ["Chicken Legs", 0.92],
     ["Chicken Thighs", 0.91]
  ],
  "Tomato": [
     ["Tomatoes", 0.99],
     ["Cherry Tomatoes", 0.89],
     ["Baby Plum Tomatoes", 0.87]
  ]
}
```

### Dateiname
```
ingredient_similarity_cache_DD-MM-YYYY-HH-MM_{METHOD}.json
```

**Generierung für alle Methoden:**
```python
for method in ["weighted", "concat", "hadamard", "max"]:
     write_JSON_similar_ingredients_fast(method)
```

---

## Validierung

### Funktion: `validate_similarity_cache()`
Prüft:
- ✅ Alle 877 Zutaten haben einen Key
- ✅ Keine leeren Listen
- ⚠️ Identifiziert problematische Fälle (z.B. zu hohe Thresholds)

---

## Probleme & Lösungen

| Problem | Ursache | Lösung |
|---------|---------|--------|
| Plural/Singular getrennt | spaCy erkennt nicht immer Lemma | Manuelle `MANUAL_CORRECTIONS` |
| "Leaves"→Lemma "Leave ≠ "Leaf" | Noise-Word-Handling | `NOISE_WORDS` Set |
| "Basil Leave" → "Basil" | Modifier dominiert | `NOISE_WORDS` Set |
| "Walnut Oil" → "Oil" | HEAD dominiert | Modifier-Vektor mit 30% gewichtet |
| JSON-Serialisierung scheitert | `numpy.float32` | Konvertierung zu Python-`float()` |

---

## Performance

- **Ohne Cache:** ~385.000 NLP-Vergleiche (877²)
- **Mit Cache:** ~877 Vergleiche (1× Initialisierung + 1× Lookup)
- **Speedup:** ~440×

In [19]:
import os
import sys

# Pfad zum übergeordneten Verzeichnis hinzufügen (damit themealdb_client gefunden wird)
sys.path.insert(0, os.path.abspath('..'))

from themealdb_client import TheMealDBClient
import json
import traceback

# Test für get_all_ingredients() Funktion

# Client initialisieren
client = TheMealDBClient()
def get_all_ingredients():
    try:
        ingredients = client.get_all_ingredients()
        ingredients = [ingredient['strIngredient'] for ingredient in ingredients if 'strIngredient' in ingredient]
        print(f"Anzahl Zutaten gefunden: {len(ingredients)}")
        print("Beispiel-Zutaten:")
        for ing in ingredients[:10]:  # Zeige die ersten 10 Zutaten
            print(f"- {ing}")
        return ingredients
    except Exception as e:
        print("Fehler beim Abrufen der Zutaten:")
        traceback.print_exc()
        return []
ALL_INGREDIENTS = get_all_ingredients()


Anzahl Zutaten gefunden: 877
Beispiel-Zutaten:
- Chicken
- Salmon
- Beef
- Pork
- Avocado
- Apple Cider Vinegar
- Asparagus
- Aubergine
- Baby Plum Tomatoes
- Bacon


In [6]:
from sentence_transformers import SentenceTransformer, util
from Levenshtein import ratio # pip install Levenshtein (misst Text-Ähnlichkeit)

# Lade ein Modell, das speziell für semantische Suche trainiert wurde
# 'all-MiniLM-L6-v2' ist klein, extrem schnell und sehr präzise
model = SentenceTransformer('all-MiniLM-L6-v2')

def is_same_ingredient(ing_a, ing_b, vector_threshold=0.75, string_threshold=0.3):
    """
    Entscheidet, ob zwei Zutaten gleich sind, indem es Bedeutung UND Schreibweise prüft.
    """
    # 1. VEKTOR CHECK (Die Bedeutung)
    # Berechnet, wie ähnlich der Kontext ist
    emb1 = model.encode(ing_a, convert_to_tensor=True)
    emb2 = model.encode(ing_b, convert_to_tensor=True)
    
    # Cosine Similarity berechnen
    semantic_score = util.cos_sim(emb1, emb2).item()
    
    # Wenn die Bedeutung schon nicht passt -> Sofort raus
    if semantic_score < vector_threshold:
        return False, semantic_score, 0.0

    # 2. STRING CHECK (Der Realitäts-Check)
    # Nutzt Levenshtein-Ratio (0 bis 1). 
    # Prüft: "Haben die Wörter gemeinsame Buchstaben-Stämme?"
    # Wir machen alles lower(), damit Groß/Klein egal ist.
    string_score = ratio(ing_a.lower(), ing_b.lower())
    
    # --- DIE ENTSCHEIDUNGSLOGIK ---
    
    # FALL A: Starke Vektor-Ähnlichkeit, aber VÖLLIG anderes Wort
    # Bsp: Broccoli (0.8) Cabbage -> String-Score ist sehr niedrig (~0.2)
    # Bsp: Potato (0.75) Tomato -> String-Score niedrig
    if string_score < string_threshold:
        # Einzige Ausnahme: Echte Synonyme wie "Aubergine" <-> "Eggplant"
        # Die müsste man hardcoden oder ein Synonym-Wörterbuch nutzen.
        # semantik Problem (Broccoli/Kohl) wird hiermit gelöst!
        return False, semantic_score, string_score
    
    # FALL B: Teil-String Match (Eines ist im anderen enthalten)
    # Bsp: "Tomato" in "Tinned Tomatoes" -> String Score mittel, aber enthalten!
    if ing_a.lower() in ing_b.lower() or ing_b.lower() in ing_a.lower():
        return True, semantic_score, string_score

    # FALL C: Hohe String-Ähnlichkeit
    # Bsp: "Tomatos" (Typo) vs "Tomatoes" -> String Score hoch
    return True, semantic_score, string_score

# --- TESTLAUF ---
pairs = [
    ("Tomato", "Tinned Tomatoes"), # Sollte True sein
    ("Broccoli", "Cabbage"),       # Sollte FALSE sein (Das war dein Problem!)
    ("Broccoli", "Cauliflower"),   # Sollte FALSE sein (Oft verwechselt)
    ("Chicken", "Beef"),           # Sollte False sein
    ("Aubergine", "Eggplant")      # Sonderfall (Wird False sein ohne Wörterbuch)
]

print(f"{'Zutat A':<15} | {'Zutat B':<15} | {'Semantik':<8} | {'String':<8} | {'MATCH?'}")
print("-" * 65)

for a, b in pairs:
    match, sem, str_sc = is_same_ingredient(a, b)
    res = "✅ JA" if match else "❌ NEIN"
    print(f"{a:<15} | {b:<15} | {sem:.4f}   | {str_sc:.4f}   | {res}")

Zutat A         | Zutat B         | Semantik | String   | MATCH?
-----------------------------------------------------------------
Tomato          | Tinned Tomatoes | 0.6950   | 0.0000   | ❌ NEIN
Broccoli        | Cabbage         | 0.6182   | 0.0000   | ❌ NEIN
Broccoli        | Cauliflower     | 0.5494   | 0.0000   | ❌ NEIN
Chicken         | Beef            | 0.6052   | 0.0000   | ❌ NEIN
Aubergine       | Eggplant        | 0.4066   | 0.0000   | ❌ NEIN


In [9]:
from sentence_transformers import SentenceTransformer, util
from Levenshtein import ratio

model = SentenceTransformer('all-MiniLM-L6-v2')

def is_same_ingredient_optimized(ing_a, ing_b):
    # 1. Vorverarbeitung
    a_lower = ing_a.lower()
    b_lower = ing_b.lower()

    # 2. Vektor-Score berechnen
    emb1 = model.encode(ing_a, convert_to_tensor=True)
    emb2 = model.encode(ing_b, convert_to_tensor=True)
    sem_score = util.cos_sim(emb1, emb2).item()
    
    # 3. String-Score berechnen
    str_score = ratio(a_lower, b_lower)
    
    # --- DYNAMISCHE LOGIK ---
    
    # FALL A: Substring-Match (Das "Tomato"-Szenario)
    # Wenn "Tomato" komplett in "Tinned Tomatoes" enthalten ist
    if a_lower in b_lower or b_lower in a_lower:
        # Hier sind wir sehr gnädig beim Vektor-Score!
        # Wir wollen nur sichergehen, dass der Kontext nicht VÖLLIG kippt 
        # (wie bei "Chicken" -> "Chicken Liver")
        # 0.55 reicht hier meistens aus.
        if sem_score > 0.55:
            return True, sem_score, str_score, "Substring Match"
            
    # FALL B: Hohe Text-Ähnlichkeit (Tippfehler / Plurale)
    if str_score > 0.8:
        return True, sem_score, str_score, "Text Match"
        
    # FALL C: Reine Synonyme (Aubergine / Eggplant)
    # Hier müssen wir STRENG sein, weil der Text nicht hilft.
    # Broccoli/Cabbage haben oft ~0.75-0.80. Wir brauchen > 0.82
    if sem_score > 0.82:
        return True, sem_score, str_score, "Semantic Match"

    return False, sem_score, str_score, "No Match"

# --- TESTLAUF ---
pairs = [
    ("Tomato", "Tinned Tomatoes"), # Score war 0.69 -> Sollte jetzt klappen (Substring Regel)
    ("Tomato", "Tomatoes"), # Score war 0.69 -> Sollte jetzt klappen (Substring Regel)
    ("Broccoli", "Cabbage"),       # Score ist hoch, aber kein Substring -> Fällt durch Fall C (zu niedrig)
    ("Aubergine", "Eggplant"),     # Hoher Vektor Score (>0.85) -> Klappt durch Fall C
    ("Chicken", "Chicken Liver"),   # Vorsicht! Substring ist True. Vektor Score muss hier geprüft werden.
    ("Chicken", "Fillet Of Steak"),   # Vorsicht! Substring ist True. Vektor Score muss hier geprüft werden.
    ("Beef", "Pork")   # Vorsicht! Substring ist True. Vektor Score muss hier geprüft werden.
]

print(f"{'A':<15} | {'B':<15} | {'Sem':<6} | {'Str':<6} | {'Resultat'}")
print("-" * 70)

for a, b in pairs:
    match, sem, str_sc, reason = is_same_ingredient_optimized(a, b)
    res = "✅" if match else "❌"
    print(f"{a:<15} | {b:<15} | {sem:.3f}  | {str_sc:.3f}  | {res} ({reason})")

A               | B               | Sem    | Str    | Resultat
----------------------------------------------------------------------
Tomato          | Tinned Tomatoes | 0.695  | 0.571  | ✅ (Substring Match)
Tomato          | Tomatoes        | 0.915  | 0.857  | ✅ (Substring Match)
Broccoli        | Cabbage         | 0.618  | 0.133  | ❌ (No Match)
Aubergine       | Eggplant        | 0.407  | 0.353  | ❌ (No Match)
Chicken         | Chicken Liver   | 0.667  | 0.700  | ✅ (Substring Match)
Chicken         | Fillet Of Steak | 0.287  | 0.182  | ❌ (No Match)
Beef            | Pork            | 0.566  | 0.000  | ❌ (No Match)


In [14]:
import spacy
from sentence_transformers import SentenceTransformer, util
from Levenshtein import ratio

# 1. SETUP: Wir brauchen BEIDE Modelle
# spaCy für die Grammatik (Wer ist der Chef im Wort?)
try:
    nlp = spacy.load("en_core_web_md") 
except:
    nlp = spacy.load("en_core_web_sm")

# Transformer für die Bedeutung (Sind Aubergine und Eggplant verwandt?)
model = SentenceTransformer('all-MiniLM-L6-v2')

# Liste von Wörtern, die den Kern nicht verändern (Container/Zustand)
SAFE_HEADS = ["leaf", "leaves", "slice", "slices", "piece", "pieces", "wedge"]

def get_head_noun(text):
    """Gibt das grammatikalische Hauptwort zurück (z.B. 'Liver' bei 'Chicken Liver')"""
    doc = nlp(text.lower())
    # Nimm das letzte Nomen oder den Root
    nouns = [t.lemma_ for t in doc if t.pos_ in ["NOUN", "PROPN"]]
    if nouns:
        return nouns[-1] # Im Englischen steht der Chef meist rechts
    return doc[-1].lemma_

def are_ingredients_equal_final(ing_a, ing_b):
    a_lower = ing_a.lower()
    b_lower = ing_b.lower()
    
    # --- SCHRITT 1: GRAMMATIK-CHECK (Der Struktur-Wächter) ---
    # Das killt "Chicken Liver", rettet aber "Tinned Tomatoes"
    
    head_a = get_head_noun(ing_a)
    head_b = get_head_noun(ing_b)
    
    # Ist eines ein Substring des anderen? (z.B. "Chicken" in "Chicken Liver")
    is_substring = a_lower in b_lower or b_lower in a_lower
    
    if is_substring:
        # GEFAHR! Wenn Strings ähnlich sind, müssen wir den Head prüfen.
        
        # Wenn die Heads UNTERSCHIEDLICH sind (Chicken != Liver)
        if head_a != head_b:
            # Ausnahme: "Basil" vs "Basil Leaves" (Head: Leaf ist harmlos)
            if head_a in SAFE_HEADS or head_b in SAFE_HEADS:
                return True, 1.0, "Safe Head Match"
                
            return False, 0.0, f"Head Mismatch ({head_a} != {head_b})"
        
        # Wenn Heads gleich sind (Tomato == Tomato), ist es sicher.
        return True, 1.0, "Substring & Head Match"

    # --- SCHRITT 2: VEKTOR-CHECK (Der Synonym-Finder) ---
    # Das findet "Aubergine", blockt aber "Broccoli/Cabbage"
    
    emb1 = model.encode(ing_a, convert_to_tensor=True)
    emb2 = model.encode(ing_b, convert_to_tensor=True)
    sem_score = util.cos_sim(emb1, emb2).item()
    
    # Aubergine/Eggplant haben im MiniLM Modell ca. 0.70 - 0.76
    # Broccoli/Cabbage haben oft nur 0.65 - 0.70
    # Wir setzen den Threshold auf 0.70, nutzen aber Levenshtein als Bremse
    
    str_score = ratio(a_lower, b_lower)
    
    # Der str_score beeinflusst den required_score dynamisch
    # Hohe Text-Ähnlichkeit = niedrigerer Semantic-Threshold nötig
    # Niedrige Text-Ähnlichkeit = höherer Semantic-Threshold nötig
    if str_score > 0.8:
        required_score = 0.65  # Text ähnlich genug → lockerer bei Semantik
    elif str_score > 0.6:
        required_score = 0.70  # Mittlere Text-Ähnlichkeit
    else:
        required_score = 0.75  # Text völlig verschieden → strenger bei Semantik
    
    if sem_score > required_score:
        return True, sem_score, "Semantic Match"
        
    return False, sem_score, "Low Similarity"

# --- DEIN HARTE TEST-SUITE ---
pairs = [
    # 1. Der Synonym-Test
    ("Aubergine", "Eggplant"),      # MUSS True sein (Semantik)
    ("Zucchini", "Courgette"),      # MUSS True sein (Semantik)
    
    # 2. Der "Teil-davon"-Fehler Test
    ("Chicken", "Chicken Liver"),   # MUSS False sein (Head: Chicken != Liver)
    ("Walnut", "Walnut Oil"),       # MUSS False sein (Head: Walnut != Oil)
    
    # 3. Der Varianten-Test
    ("Tomato", "Tinned Tomatoes"),  # MUSS True sein (Head: Tomato == Tomato)
    ("Basil", "Basil Leaves"),      # MUSS True sein (Ausnahme-Liste)
    ("Basil", "Cloves"),      # MUSS True sein (Ausnahme-Liste)
    ("Cumin", "Cumin Seeds"),      # MUSS True sein (Ausnahme-Liste)
    
    # 4. Der "Falsche Freunde" Test
    ("Broccoli", "Cabbage"),        # MUSS False sein (Score zu niedrig)
    ("Potato", "Sweet Potato")      # MUSS False sein (Head: Potato != Potato? Vorsicht!)
]

print(f"{'A':<15} | {'B':<15} | {'Resultat':<10} | {'Grund'}")
print("-" * 60)

for a, b in pairs:
    match, score, reason = are_ingredients_equal_final(a, b)
    icon = "✅" if match else "❌"
    print(f"{a:<15} | {b:<15} | {icon} {match:<6} | {reason} (Score: {score:.2f})")

A               | B               | Resultat   | Grund
------------------------------------------------------------
Aubergine       | Eggplant        | ❌ 0      | Low Similarity (Score: 0.41)
Zucchini        | Courgette       | ❌ 0      | Low Similarity (Score: 0.46)
Chicken         | Chicken Liver   | ❌ 0      | Head Mismatch (chicken != liver) (Score: 0.00)
Walnut          | Walnut Oil      | ❌ 0      | Head Mismatch (walnut != oil) (Score: 0.00)
Tomato          | Tinned Tomatoes | ✅ 1      | Substring & Head Match (Score: 1.00)
Basil           | Basil Leaves    | ✅ 1      | Substring & Head Match (Score: 1.00)
Basil           | Cloves          | ❌ 0      | Low Similarity (Score: 0.41)
Cumin           | Cumin Seeds     | ❌ 0      | Head Mismatch (cumin != seed) (Score: 0.00)
Broccoli        | Cabbage         | ❌ 0      | Low Similarity (Score: 0.62)
Potato          | Sweet Potato    | ✅ 1      | Substring & Head Match (Score: 1.00)


In [ ]:
# Globale Liste an "harmlosen" Köpfen (Container/Zustände),
# die den eigentlichen Kern einer Zutat nicht verändern.
# Beispiel: "Basil Leaves" soll zu "Basil" passen.
SAFE_HEADS = ["leaf", "leaves", "slice", "slices", "piece", "pieces", "wedge", "clove", "cloves", "seed", "seeds"]

class IngredientSimilarityChecker:
    """
    Bewertet Zutaten-Paare semantisch (SentenceTransformer) + strukturell (spaCy-Head).
    Optimiert für Wiederverwendung: Modelle + Embeddings werden nur einmal geladen.
    """

    def __init__(self, ingredient_list):
        """
        Initialisiert die Klasse, lädt Modelle und erstellt Embeddings.

        Schritte:
        1) spaCy-Modell laden (für Grammatik/Head-Erkennung)
        2) SentenceTransformer laden (für semantische Ähnlichkeit)
        3) Levenshtein-Ratio vorbereiten (für String-Ähnlichkeit)
        4) Embeddings für die Zutatenliste vorab berechnen
        5) Head-Nouns für die Zutatenliste vorab berechnen
        """
        import spacy
        from sentence_transformers import SentenceTransformer, util
        from Levenshtein import ratio

        # Zutatenliste speichern
        self.ingredient_list = ingredient_list

        # 1) spaCy-Modelle laden (md bevorzugt für Vektoren)
        try:
            self.nlp = spacy.load("en_core_web_md")
        except:
            self.nlp = spacy.load("en_core_web_sm")

        # 2) Transformer-Modell für semantische Ähnlichkeit
        self.model = SentenceTransformer('all-MiniLM-L6-v2')

        # 3) Utilities zwischenspeichern
        self.util = util
        self.ratio = ratio

        # Normalisierte Zutatenliste (einheitliche Keys)
        normalized_ingredients = [self._normalize_text(ing) for ing in ingredient_list]
        normalized_unique = list(dict.fromkeys(normalized_ingredients))

        # 4) Embeddings für alle Zutaten vorberechnen (einmalig)
        self.embeddings = {
            ing_norm: self.model.encode(ing_norm, convert_to_tensor=True)
            for ing_norm in normalized_unique
        }

        # 5) Head-Nouns für alle Zutaten vorberechnen (einmalig)
        self.head_nouns = {
            ing_norm: self._compute_head_noun(ing_norm)
            for ing_norm in normalized_unique
        }

    def _normalize_text(self, text):
        """
        Normalisiert Zutaten-Strings konsistent für Cache-Keys.
        - lower()
        - Mehrfache Spaces entfernen
        - Leading/Trailing Spaces trimmen
        """
        return " ".join(text.lower().split())

    def _compute_head_noun(self, text):
        """
        Berechnet das grammatikalische Kopf-Nomen eines Ausdrucks.
        Beispiel: "Chicken Liver" -> "liver" (im Englischen steht der Head meist rechts).
        """
        doc = self.nlp(text.lower())

        # Alle Nomen/Proper-Nomen sammeln
        nouns = [t.lemma_ for t in doc if t.pos_ in ["NOUN", "PROPN"]]

        # Falls vorhanden, nimm das letzte Nomen als Head
        if nouns:
            return nouns[-1]

        # Fallback: letztes Token lemmatisiert
        return doc[-1].lemma_

    def get_head_noun(self, text):
        """
        Liefert das Head-Nomen aus dem Cache, falls vorhanden.
        Fallback: berechnet es on-the-fly für unbekannte Zutaten.
        """
        key = self._normalize_text(text)
        cached = self.head_nouns.get(key)
        if cached is not None:
            return cached
        return self._compute_head_noun(key)

    def _get_embedding(self, ingredient):
        """
        Liefert ein vorhandenes Embedding aus dem Cache.
        Wenn die Zutat nicht im Cache ist, wird es on-the-fly erzeugt.
        """
        key = self._normalize_text(ingredient)
        emb = self.embeddings.get(key)
        if emb is None:
            emb = self.model.encode(key, convert_to_tensor=True)
        return emb

    def similarity(self, ing_a, ing_b):
        """
        Hauptlogik für den Vergleich zweier Zutaten.

        Schritte:
        1) Strings normalisieren
        2) Grammatik-Check: Head-Nomen vergleichen
        3) Substring-Regel behandeln ("Tomato" in "Tinned Tomatoes")
        4) Semantik-Score via Transformer berechnen
        5) String-Score via Levenshtein berechnen
        6) Dynamischen Threshold bestimmen und entscheiden
        """
        # 1) Normalisierung für String-Vergleiche
        a_norm = self._normalize_text(ing_a)
        b_norm = self._normalize_text(ing_b)

        # 2) Head-Nomen bestimmen
        head_a = self.get_head_noun(a_norm)
        head_b = self.get_head_noun(b_norm)

        # 3) Substring-Prüfung (A ist Teil von B oder umgekehrt)
        is_substring = a_norm in b_norm or b_norm in a_norm
        if is_substring:
            # Wenn Heads unterschiedlich sind, ist es oft KEIN Match
            if head_a != head_b:
                # Ausnahme: SAFE_HEADS erlauben Varianten wie "Basil Leaves"
                if head_a in SAFE_HEADS or head_b in SAFE_HEADS:
                    return True, 1.0, "Safe Head Match"
                return False, 0.0, f"Head Mismatch ({head_a} != {head_b})"

            # Heads gleich -> sicherer Match
            return True, 1.0, "Substring & Head Match"

        # 4) Semantische Ähnlichkeit (Transformer)
        emb1 = self._get_embedding(a_norm)
        emb2 = self._get_embedding(b_norm)
        sem_score = float(self.util.cos_sim(emb1, emb2).item())

        # 5) String-Ähnlichkeit (Levenshtein)
        str_score = self.ratio(a_norm, b_norm)

        # 6) Dynamischer Threshold
        # Hoher String-Score -> Semantik darf etwas niedriger sein
        if str_score > 0.8:
            required_score = 0.65
        elif str_score > 0.6:
            required_score = 0.70
        else:
            required_score = 0.75

        # Entscheidung
        if sem_score > required_score:
            return True, sem_score, "Semantic Match"
        return False, sem_score, "Low Similarity"

# Beispielnutzung:
ingredients = [
    "Aubergine", "Eggplant", "Broccoli", "Cabbage",
    "Tomato", "Tinned Tomatoes", "Chicken", "Chicken Liver",
    "Basil", "Basil Leaves"
]
checker = IngredientSimilarityChecker(ingredients)

pairs = [
    ("Aubergine", "Eggplant"),
    ("Broccoli", "Cabbage"),
    ("Tomato", "Tinned Tomatoes"),
    ("Chicken", "Chicken Liver"),
    ("Basil", "Basil Leaves"),
    ("Basil", "fresh Basil Leaves"),
    ("Tomato", "tomato sauce")
]

print(f"{'A':<15} | {'B':<15} | {'Resultat':<10} | {'Grund'}")
print("-" * 60)
for a, b in pairs:
    match, score, reason = checker.similarity(a, b)
    icon = "✅" if match else "❌"
    print(f"{a:<15} | {b:<15} | {icon} {match:<6} | {reason} (Score: {score:.2f})")


A               | B               | Resultat   | Grund
------------------------------------------------------------
Aubergine       | Eggplant        | ❌ 0      | Low Similarity (Score: 0.41)
Broccoli        | Cabbage         | ❌ 0      | Low Similarity (Score: 0.62)
Tomato          | Tinned Tomatoes | ✅ 1      | Substring & Head Match (Score: 1.00)
Chicken         | Chicken Liver   | ❌ 0      | Head Mismatch (chicken != liver) (Score: 0.00)
Basil           | Basil Leaves    | ✅ 1      | Substring & Head Match (Score: 1.00)
Basil           | fresh Basil Leaves | ✅ 1      | Substring & Head Match (Score: 1.00)
Tomato          | tomato sauce    | ❌ 0      | Head Mismatch (tomato != sauce) (Score: 0.00)


In [ ]:
# JSON-ähnliches Dict mit Gründen (nur wenn Match == True)
# Aufbau:
# {
#   "Zutat A": [
#       ["Zutat B", "Substring & Head Match (Score: 1.00)"],
#       ["Zutat C", "Semantic Match (Score: 0.78)"]
#   ]
# }

# Beispiel-Paare (ersetze oder generiere dynamisch)
pairs = [
    ("Aubergine", "Eggplant"),
    ("Broccoli", "Cabbage"),
    ("Tomato", "Tinned Tomatoes"),
    ("Chicken", "Chicken Liver"),
    ("Basil", "Basil Leaves"),
    ("Basil", "fresh Basil Leaves"),
    ("Tomato", "tomato sauce")
]

# Dict füllen
similarity_dict = {}
for a, b in pairs:
    match, score, reason = checker.similarity(a, b)
    if match:
        if a not in similarity_dict:
            similarity_dict[a] = []
        similarity_dict[a].append([b, f"{reason} (Score: {score:.2f})"])

# Ausgabe (optional)
import json
print(json.dumps(similarity_dict, indent=2, ensure_ascii=False))

In [20]:
def build_ingredient_similarity_json():
    """
    Erstellt ein Ähnlichkeits-Dict für alle Zutaten aus ALL_INGREDIENTS.
    Vergleicht jede Zutat mit allen anderen Zutaten.
    Speichert nur Matches (match==True) mit Gründen in eine JSON-Datei.
    
    Format:
    {
        "Zutat A": [
            ["Zutat B", "Substring & Head Match (Score: 1.00)"],
            ["Zutat C", "Semantic Match (Score: 0.78)"]
        ]
    }
    
    Dateiname: ingredient_similarity_json_DD-MM-YYYY-HH-MM.json
    """
    from datetime import datetime
    import json
    
    print(f"Initialisiere IngredientSimilarityChecker für {len(ALL_INGREDIENTS)} Zutaten...")
    checker_full = IngredientSimilarityChecker(ALL_INGREDIENTS)
    
    similarity_dict = {}
    
    print(f"\nVergleiche alle Zutaten-Paare...")
    total_pairs = len(ALL_INGREDIENTS)
    
    for idx, ingredient_a in enumerate(ALL_INGREDIENTS):
        if idx % 50 == 0:
            print(f"Fortschritt: {idx}/{total_pairs}")
        
        # Vergleiche mit allen anderen Zutaten
        for ingredient_b in ALL_INGREDIENTS:
            # Ähnlichkeit berechnen
            match, score, reason = checker_full.similarity(ingredient_a, ingredient_b)
            
            # Nur speichern, wenn Match == True
            if match:
                if ingredient_a not in similarity_dict:
                    similarity_dict[ingredient_a] = []
                similarity_dict[ingredient_a].append([ingredient_b, f"{reason} (Score: {score:.2f})"])
    
    # Dateiname mit Zeitstempel erstellen
    timestamp = datetime.now().strftime("%d-%m-%Y-%H-%M")
    output_file = f"ingredient_similarity_semantic_{timestamp}.json"
    
    # JSON speichern
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(similarity_dict, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Fertig! Datei gespeichert: {output_file}")
    print(f"Zutaten mit Matches: {len(similarity_dict)}")
    print(f"Gesamte Matches: {sum(len(matches) for matches in similarity_dict.values())}")
    
    return similarity_dict

# Ausführen
similarity_results = build_ingredient_similarity_json()


Initialisiere IngredientSimilarityChecker für 877 Zutaten...

Vergleiche alle Zutaten-Paare...
Fortschritt: 0/877
Fortschritt: 50/877
Fortschritt: 100/877
Fortschritt: 150/877


KeyboardInterrupt: 

In [ ]:
import numpy as np
import spacy

METHOD = "max"

# --- Schneller IngredientPool-Index für NLP-Vergleiche ---

# Vektoren funktionieren nur mit md oder lg!
try:
    nlp_score   = spacy.load("en_core_web_md")
    nlp_head    = spacy.load("en_core_web_trf")
except:
    print("Bitte lade das Medium-Modell: python -m spacy download en_core_web_md")
    nlp_score = spacy.load("en_core_web_sm") # Fallback (wird aber Warnung werfen)
    
# 1. NOISE (Ignorieren wir komplett)
NOISE_WORDS = {
    "leaves", "leaf", "seed", "seeds", "nuts", "yolks"
}

MANUAL_CORRECTIONS = {"leaves": "leaf", "leave": "leaf", "nuts": "nut", "yolks": "yolk"}


class IngredientTokenCache:
    """
    Cache für NLP-Token aller Zutaten aus dem Pool.
    Berechnet einmalig alle Token für schnellere Vergleiche.
    """
    
    def __init__(self, ingredient_pool, method=METHOD):
        """
        Initialisiert den Token-Cache für alle Zutaten.
        
        Args:
            ingredient_pool: Liste aller Zutaten
            method: Kombinationsmethode für Head + Modifier-Tokens
        """
        print(f"Initialisiere Token-Cache für {len(ingredient_pool)} Zutaten...")
        self.token_dict = {}
        self.combine_tokens = {}
        self.method = method
        
        for idx, ingredient in enumerate(ingredient_pool):
            if idx % 100 == 0:
                print(f"  Fortschritt: {idx}/{len(ingredient_pool)}")
            self.token_dict[ingredient] = self._get_head_and_mod_token(ingredient)
            self.combine_tokens[ingredient] = self._combine_tokens(*self.token_dict[ingredient])
        print("Token-Cache bereit!")
    
    # ========== GRAMMATIK-ANALYSE ==========
    
    def _analyze_grammar(self, text):
        """
        Analysiert die grammatikalische Struktur einer Zutat.
        Extrahiert: HEAD (Hauptwort) und MODIFIER (Beschreibungen).
        
        Beispiel: "walnut oil" → HEAD="oil", MODIFIER="walnut"
        
        Args:
            text: Zutatenstext (z.B. "walnut oil")
        
        Returns:
            tuple: (modifiers_combined, head_word)
                - modifiers_combined: " "-separierte Modifier (z.B. "walnut")
                - head_word: Hauptwort nach manuellen Korrektionen
        """
        doc = nlp_head(text.lower())
        
        # 1. Den grammatikalischen Kern finden (ROOT)
        # Das Wort, das von keinem anderen abhängt
        head_token = [t for t in doc if t.head == t][0]
        
        # 2. Modifiers sammeln (erweiterte Information)
        # z.B. Nomen, die den Typ bestimmen (Walnut → Oil)
        modifiers = []
        
        for child in head_token.children:
            # 3. Ignoriere Präpositionen (prep) wie "of" in "Cup of Walnuts"
            if child.dep_ == "prep":
                continue
            
            # Manuelle Korrektur für Modifier (z.B. "leave" → "leaf")
            modifiers.append(MANUAL_CORRECTIONS.get(child.text, child.lemma_))
        
        # Sortiere Modifier alphabetisch für Konsistenz
        modifiers.sort()
        modifiers_combined = " ".join(modifiers)
        
        # 3. Manuelle Korrektionen für HEAD (z.B. "leaves" → "leaf")
        found_head = MANUAL_CORRECTIONS.get(head_token.text, head_token.lemma_)
        
        # Spezialfall: Wenn HEAD in NOISE_WORDS, dann wechsel zu MODIFIER
        if found_head in NOISE_WORDS:
            found_head = modifiers_combined
            modifiers_combined = ""
        
        return (modifiers_combined, found_head)
    
    # ========== TOKEN-KOMBINATION ==========
    
    def _combine_tokens(self, head_token, mod_token, weight_head=0.7, weight_mod=0.3):
        """
        Kombiniert HEAD-Token und MODIFIER-Token zu einem Vektor.
        
        Methoden:
            - "weighted": Gewichteter Durchschnitt (Standard)
            - "weighted7030": 70% HEAD + 30% MODIFIER
            - "weighted5050": 50% HEAD + 50% MODIFIER
            - "concat": Vektoren konkatenieren (doppelte Länge!)
            - "hadamard": Element-weise Multiplikation
            - "max": Element-weises Maximum (Standard)
        
        Args:
            head_token: spaCy Token für HEAD
            mod_token: spaCy Token für MODIFIER
            weight_head: Gewicht für HEAD (Standard 0.7)
            weight_mod: Gewicht für MODIFIER (Standard 0.3)
        
        Returns:
            numpy.ndarray: Der kombinierte Vektor
        """
        # Wenn keine Modifier: Gib HEAD-Vektor zurück
        if mod_token.text == "":
            return head_token.vector
        
        if self.method == "weighted":
            # Gewichteter Durchschnitt (70% HEAD + 30% MODIFIER)
            combined_vector = (
                head_token.vector * weight_head + 
                mod_token.vector * weight_mod
            ) / (weight_head + weight_mod)
        
        elif self.method == "weighted7030":
            # Explizit 70% HEAD + 30% MODIFIER
            combined_vector = (
                head_token.vector * 0.7 + 
                mod_token.vector * 0.3
            ) / 1.0
        
        elif self.method == "weighted5050":
            # 50% HEAD + 50% MODIFIER
            combined_vector = (
                head_token.vector * 0.5 + 
                mod_token.vector * 0.5
            ) / 1.0
        
        elif self.method == "concat":
            # Beide Vektoren aneinanderhängen (600D statt 300D)
            # Vorteil: Keine Information geht verloren
            # Nachteil: Doppelte Länge
            combined_vector = np.concatenate([head_token.vector, mod_token.vector])
        
        elif self.method == "hadamard":
            # Element-weise Multiplikation (Hadamard-Produkt)
            # Vorteil: Betont gemeinsame Features
            # Nachteil: Kann Nullen erzeugen
            combined_vector = head_token.vector * mod_token.vector
        
        elif self.method == "max":
            # Element-weises Maximum
            # Vorteil: Behält stärkste Features beider Vektoren
            combined_vector = np.maximum(head_token.vector, mod_token.vector)
        
        else:
            raise ValueError(
                f"Unbekannte Methode: {self.method}. "
                f"Verwende 'weighted', 'weighted7030', 'weighted5050', "
                f"'concat', 'hadamard' oder 'max'"
            )
        
        return combined_vector
    
    # ========== TOKEN-GENERIERUNG ==========
    
    def _get_head_and_mod_token(self, ingredient):
        """
        Konvertiert eine Zutat in NLP-Tokens (Head + Modifiers).
        
        Args:
            ingredient: Zutatenstext
        
        Returns:
            tuple: (head_token, modifier_token)
        """
        modifiers_combined, ingredient_head = self._analyze_grammar(ingredient)
        return nlp_score(ingredient_head), nlp_score(modifiers_combined)
    
    def _get_head_and_mod_token_combined(self, ingredient):
        """
        Konvertiert eine Zutat in einen kombinierten Token.
        
        Args:
            ingredient: Zutatenstext
        
        Returns:
            numpy.ndarray: Der kombinierte Vektor (HEAD + MODIFIER)
        """
        modifiers_combined, ingredient_head = self._analyze_grammar(ingredient)
        
        # Kombiniere HEAD + MODIFIER zu einem neuen Vektor
        combined_token = self._combine_tokens(
            nlp_score(ingredient_head), 
            nlp_score(modifiers_combined)
        )
        
        return combined_token
    
    # ========== ÄHNLICHKEITS-VERGLEICH ==========
    
    def check_similarity(self, base_ingredient, threshold_head=0.8, 
                        threshold_mod=0.8, debug=False):
        """
        Vergleicht eine Zutat mit allen gecachten Zutaten.
        Bewertet HEAD und MODIFIER separat.
        
        Args:
            base_ingredient: Zutat zum Vergleichen
            threshold_head: Mindest-Score für HEAD (0.0-1.0)
            threshold_mod: Mindest-Score für MODIFIER (0.0-1.0)
            debug: Verbose Output
        
        Returns:
            dict: {base_ingredient: [(zutat, score_head, score_mod), ...]}
        """
        base_token_head, base_token_mods = self._get_head_and_mod_token(base_ingredient)
        results = []
        
        for comp_ingredient, (comp_token_head, comp_token_mods) in self.token_dict.items():
            # Berechne HEAD-Ähnlichkeit
            score_head = base_token_head.similarity(comp_token_head)
            
            # Berechne MODIFIER-Ähnlichkeit (fallback: -10 wenn leer)
            score_mods = -10
            if base_token_mods.text and comp_token_mods.text:
                score_mods = base_token_mods.similarity(comp_token_mods)
            
            if debug:
                print(f"{comp_ingredient:<30} | Head: {score_head:.4f} | Mod: {score_mods:.4f}")
            
            # Filter: HEAD über Threshold UND (MODIFIER über Threshold ODER leer)
            if (score_head > threshold_head and score_head <= 1.0) and \
               (score_mods > threshold_mod or score_mods == -10):
                results.append((comp_ingredient, float(score_head), float(score_mods)))
        
        # Sortiere nach HEAD-Score absteigend
        results.sort(key=lambda x: x[1], reverse=True)
        
        if debug:
            print("\n" + "="*60)
            for comp_ingredient, score_head, score_mods in results:
                bar_head = "█" * int(score_head * 10)
                bar_mods = "█" * int(max(score_mods, 0) * 10)
                print(f"{comp_ingredient:<20} | {score_head:.4f} {bar_head}  | {score_mods:.4f} {bar_mods}")
        
        return {base_ingredient: results}
    
    def check_similarity_combined(self, base_ingredient, threshold=0.8, debug=False):
        """
        Vergleicht eine Zutat mit allen gecachten Zutaten.
        Verwendet den kombinierten Token (HEAD + MODIFIER).
        
        Args:
            base_ingredient: Zutat zum Vergleichen
            threshold: Mindest-Score (0.0-1.0)
            debug: Verbose Output
        
        Returns:
            dict: {base_ingredient: [(zutat, combined_score), ...]}
        """
        base_token_combined = self._get_head_and_mod_token_combined(base_ingredient)
        results = []
        
        for comp_ingredient in self.combine_tokens.keys():
            comp_token_combined = self.combine_tokens[comp_ingredient]
            
            # Berechne Cosinus-Ähnlichkeit zwischen Vektoren
            combined_score = np.dot(base_token_combined, comp_token_combined) / (
                np.linalg.norm(base_token_combined) * 
                np.linalg.norm(comp_token_combined)
            )
            
            if debug:
                print(f"{comp_ingredient:<30} | Combined Score: {combined_score:.4f}")
            
            # Filter: Score über Threshold und <= 1.0
            if combined_score > threshold and combined_score <= 1.0:
                # Konvertiere numpy.float32 zu Python-float für JSON-Serialisierung
                results.append((comp_ingredient, float(combined_score)))
        
        # Sortiere nach Score absteigend
        results.sort(key=lambda x: x[1], reverse=True)
        
        if debug:
            print("\n" + "="*60)
            for comp_ingredient, combined_score in results:
                bar = "█" * int(combined_score * 10)
                print(f"{comp_ingredient:<20} | {combined_score:.4f} {bar}")
        
        return {base_ingredient: results}


# Beispielnutzung:
# cache = IngredientTokenCache(ALL_INGREDIENTS)
# result = cache.check_similarity_combined("Tomato", threshold=0.75, debug=True)


In [4]:
import json
from datetime import datetime

def write_JSON_similar_ingredients_fast(method=METHOD):
    """
    Erstellt eine JSON-Datei mit ähnlichen Zutaten für alle Zutaten.
    Nutzt IngredientTokenCache für schnellere Verarbeitung.
    Format: { "Zutat": ["Ähnliche1", "Ähnliche2", ...] }
    Dateiname: ingredient_similarity_cache_DD-MM-YYYY-HH-MM.json
    """
    
    # Cache einmalig initialisieren (das dauert, spart aber enorm Zeit danach)
    cache = IngredientTokenCache(ALL_INGREDIENTS, method=method)
    
    similarity_data = {}
    
    print(f"\nVerarbeite {len(ALL_INGREDIENTS)} Zutaten mit Methode '{method}'...")
    
    for index, ingredient in enumerate(ALL_INGREDIENTS):
        if index % 50 == 0:
            print(f"Fortschritt: {index}/{len(ALL_INGREDIENTS)}")
        
        # Ähnliche Zutaten finden (Head + Modifiers)
        if False:
            result = cache.check_similarity(ingredient, thresholdHead=0.9, thresholdMod=0.8, debug=False)
        
        # Ähnliche Zutaten finden (kombinierter Token)    
        else:
            result = cache.check_similarity_combined(ingredient, threshold=0.85, debug=False)
        
        # Nur die Zutaten-Namen extrahieren
        if ingredient in result and result[ingredient]:
            similarity_data[ingredient] = list(result[ingredient])
    
    # Dateiname mit Datum erstellen
    timestamp = datetime.now().strftime("%d-%m-%Y-%H-%M")
    output_file = f"ingredient_similarity_cache_{timestamp}_{method}.json"
    
    # JSON speichern (nur das Dictionary)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(similarity_data, f, indent=2, ensure_ascii=False)
    
    print(f"\nFertig! Datei gespeichert: {output_file}")
    print(f"Zutaten mit Matches: {len(similarity_data)}")
    
    return similarity_data

# Funktion ausführen (jetzt mit Cache - viel schneller!)
for method in ["weighted", "concat", "hadamard", "max"]:
    similarity_results = write_JSON_similar_ingredients_fast(method)

Initialisiere Token-Cache für 877 Zutaten...
  Fortschritt: 0/877


d:\GitRepo\PKI_Projekt_Gruppe_B1_4\.venvRezept\lib\site-packages\thinc\shims\pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


  Fortschritt: 100/877
  Fortschritt: 200/877
  Fortschritt: 300/877
  Fortschritt: 400/877
  Fortschritt: 500/877
  Fortschritt: 600/877
  Fortschritt: 700/877
  Fortschritt: 800/877
Token-Cache bereit!

Verarbeite 877 Zutaten mit Methode 'weighted'...
Fortschritt: 0/877


C:\Users\maxi9\AppData\Local\Temp\ipykernel_22760\1308947194.py:125: RuntimeWarning: invalid value encountered in scalar divide
  combined_score = np.dot(baseTokenCombined, comparableTokenCombined) / (np.linalg.norm(baseTokenCombined) * np.linalg.norm(comparableTokenCombined))


Fortschritt: 50/877
Fortschritt: 100/877
Fortschritt: 150/877
Fortschritt: 200/877
Fortschritt: 250/877
Fortschritt: 300/877
Fortschritt: 350/877
Fortschritt: 400/877
Fortschritt: 450/877
Fortschritt: 500/877
Fortschritt: 550/877
Fortschritt: 600/877
Fortschritt: 650/877
Fortschritt: 700/877
Fortschritt: 750/877
Fortschritt: 800/877
Fortschritt: 850/877

Fertig! Datei gespeichert: ingredient_similarity_cache_30-01-2026-22-23_weighted.json
Zutaten mit Matches: 828
Initialisiere Token-Cache für 877 Zutaten...
  Fortschritt: 0/877
  Fortschritt: 100/877
  Fortschritt: 200/877
  Fortschritt: 300/877
  Fortschritt: 400/877
  Fortschritt: 500/877
  Fortschritt: 600/877
  Fortschritt: 700/877
  Fortschritt: 800/877
Token-Cache bereit!

Verarbeite 877 Zutaten mit Methode 'concat'...
Fortschritt: 0/877


ValueError: shapes (300,) and (600,) not aligned: 300 (dim 0) != 600 (dim 0)

In [4]:
def validate_similarity_cache(json_file, all_ingredients_list):
    """
    Validiert die erstellte JSON-Datei:
    - Prüft, ob jede Zutat einen Key hat
    - Prüft, ob die Listen nicht leer sind
    """
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            cache_data = json.load(f)
    except Exception as e:
        print(f"Fehler beim Lesen der Datei: {e}")
        return False
    
    print(f"Validiere {json_file}...")
    print(f"Zutaten in ALL_INGREDIENTS: {len(all_ingredients_list)}")
    print(f"Keys in JSON: {len(cache_data)}")
    
    missing_keys = []
    empty_lists = []
    
    # Prüfe, ob alle Zutaten einen Key haben
    for ingredient in all_ingredients_list:
        if ingredient not in cache_data:
            missing_keys.append(ingredient)
        elif not cache_data[ingredient]:  # Liste ist leer
            empty_lists.append(ingredient)
    
    # Ausgabe der Ergebnisse
    print("\n--- VALIDIERUNGSERGEBNISSE ---")
    
    if missing_keys:
        print(f"\n❌ FEHLER: {len(missing_keys)} Zutaten ohne Key:")
        for ing in missing_keys[:10]:  # Zeige nur die ersten 10
            print(f"  - {ing}")
        if len(missing_keys) > 10:
            print(f"  ... und {len(missing_keys) - 10} mehr")
    else:
        print("✅ Alle Zutaten haben einen Key")
    
    if empty_lists:
        print(f"\n⚠️ WARNUNG: {len(empty_lists)} Zutaten haben leere Listen:")
        for ing in empty_lists[:10]:  # Zeige nur die ersten 10
            print(f"  - {ing}")
        if len(empty_lists) > 10:
            print(f"  ... und {len(empty_lists) - 10} mehr")
    else:
        print("✅ Keine leeren Listen gefunden")
    
    # Zusammenfassung
    success = len(missing_keys) == 0 and len(empty_lists) == 0
    print(f"\n{'✅ VALIDIERUNG ERFOLGREICH' if success else '❌ VALIDIERUNG FEHLGESCHLAGEN'}")
    
    return success

# # Beispielnutzung (nach Ausführung von write_JSON_similar_ingredients_fast()):
validate_similarity_cache("used_similarity_cache/ingredient_similarity_cache_30-01-2026-15-48_max.json", ALL_INGREDIENTS)


Validiere used_similarity_cache/ingredient_similarity_cache_30-01-2026-15-48_max.json...
Zutaten in ALL_INGREDIENTS: 877
Keys in JSON: 744

--- VALIDIERUNGSERGEBNISSE ---

❌ FEHLER: 133 Zutaten ohne Key:
  - Bacon
  - Bay Leaf
  - Bay Leaves
  - Borlotti Beans
  - Bramley Apples
  - Brandy
  - Butter
  - Cacao
  - Cayenne Pepper
  - Challots
  ... und 123 mehr
✅ Keine leeren Listen gefunden

❌ VALIDIERUNG FEHLGESCHLAGEN


False

In [ ]:
# --- TEST mit Cache ---
cache = IngredientTokenCache(ALL_INGREDIENTS)
result = cache.check_similarity("Cucumber", thresholdHead=0.8, debug=False)
print(result)


Initialisiere Token-Cache für 877 Zutaten...
  Fortschritt: 0/877
  Fortschritt: 100/877
  Fortschritt: 200/877
  Fortschritt: 300/877
  Fortschritt: 400/877
  Fortschritt: 500/877
  Fortschritt: 600/877
  Fortschritt: 700/877
  Fortschritt: 800/877
Token-Cache bereit!
{'Cucumber': [('Cucumber', 1.0, -10), ('Persian Cucumber', 1.0, -10)]}


C:\Users\maxi9\AppData\Local\Temp\ipykernel_29376\2219725144.py:27: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  scoreHead = baseTokenHead.similarity(comparableTokenHead)


In [ ]:
result = cache.check_similarity("Cucumber", thresholdHead=0.8, debug=False)
print(result)

{'Cucumber': [('Cucumber', 1.0, -10), ('Persian Cucumber', 1.0, -10)]}


C:\Users\maxi9\AppData\Local\Temp\ipykernel_29376\2219725144.py:27: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  scoreHead = baseTokenHead.similarity(comparableTokenHead)
